# Code mẫu — Zero-shot

Zero-shot dùng những gì model đã học để chấm cả những lớp chưa từng có nhãn train. Dù vậy, vẫn phải kiểm ba thứ: vector đại diện cho mỗi lớp, tập lớp ứng viên, và cách tính điểm. Hai ví dụ đầu dùng vector nhỏ để kiểm phép hình học theo kiểu CLIP. Hai ví dụ sau chấm điểm chuỗi token, kết thúc bằng một model bigram tự học.

## Ví dụ 1: Hai mô tả cho mỗi lớp, năm truy vấn

Mỗi lớp có hai vector mô tả, mỗi vector hai chiều. Ta đưa từng mô tả về độ dài 1, lấy trung bình theo lớp, rồi đưa lại về độ dài 1. Vector thu được gọi là prototype của lớp. Các vector đều tự đặt. Tên `san` và `phong` chỉ để đọc kết quả cho dễ, không phải embedding thật của CLIP.

In [ ]:
import numpy as np

def unit_rows(values):
    norms = np.linalg.norm(values, axis=-1, keepdims=True)
    if np.any(norms == 0):
        raise ValueError("khong normalize vector 0")
    return values / norms

templates = np.array([[[2., 0.], [3., 1.]],
                      [[0., 2.], [1., 3.]]])  # Shape (lớp, mô tả, tọa độ).
prototypes = unit_rows(unit_rows(templates).mean(axis=1))
queries = unit_rows(np.array([[3., 1.], [1., 3.], [2., 0.],
                              [0., 2.], [1., 1.]]))
names = np.array(["san", "phong"])

def classify_batches(values, batch_size):
    return np.concatenate([values[i:i + batch_size] @ prototypes.T
                           for i in range(0, len(values), batch_size)])

scores = classify_batches(queries, 2)
reference = np.array([[sum(q[d] * p[d] for d in range(2))
                       for p in prototypes] for q in queries])
assert np.allclose(scores, reference)
assert np.allclose(scores, classify_batches(queries, 5))
assert np.allclose(np.linalg.norm(prototypes, axis=1), 1)
print("prototype:", prototypes.round(3).tolist())
print("shape diem:", scores.shape)
print("cosine:", scores.round(3).tolist())
print("nhan:", names[scores.argmax(axis=1)].tolist())

prototype: [[0.987, 0.16], [0.16, 0.987]]
shape diem: (5, 2)
cosine: [[0.987, 0.464], [0.464, 0.987], [0.987, 0.16], [0.16, 0.987], [0.811, 0.811]]
nhan: ['san', 'phong', 'san', 'phong', 'san']


Bảng điểm có năm hàng và hai cột: mỗi hàng là một truy vấn, mỗi cột là một lớp. Vì mọi vector đều dài 1, tích vô hướng chính là cosine. Chạy theo batch 2, 2, 1 vẫn đủ năm hàng, và khớp với cách tính bằng vòng lặp thường.

Hãy nhìn hàng cuối. Truy vấn `(1, 1)` nằm đúng giữa hai lớp, nên hai điểm bằng nhau, 0.811. `argmax` chọn cột đầu chỉ vì NumPy phá hòa theo vị trí. Không có bằng chứng nào cho thấy `san` hợp hơn. Khi điểm hòa hoặc rất sát nhau, kết quả gần như chỉ là may rủi.

## Ví dụ 2: Temperature đổi độ chắc chắn, tập lớp đổi ý nghĩa

Truy vấn `(0.8, 0.6)` so với hai prototype `(1, 0)` và `(0, 1)`, cho cosine 0.8 và 0.6. Ta chia cosine cho temperature trước khi softmax. Temperature nhỏ làm phân phối dồn về lớp thắng, nhưng không đổi lớp thắng. Sau đó ta thử một truy vấn có cosine âm với cả hai lớp, để thấy giới hạn của việc chỉ chọn trong một tập lớp cố định.

In [ ]:
import numpy as np

def softmax(rows):
    values = np.exp(rows - rows.max(axis=1, keepdims=True))
    return values / values.sum(axis=1, keepdims=True)

prototypes = np.eye(2)
query = np.array([[0.8, 0.6]])
cosine = query @ prototypes.T
for temperature in [1.0, 0.2]:
    probability = softmax(cosine / temperature)
    assert probability.argmax(1)[0] == 0
    print("temperature:", temperature, "softmax:", probability.round(4).tolist())

outside = np.array([[-1., -0.1]])
outside /= np.linalg.norm(outside)
old_scores = outside @ prototypes.T
old_probability = softmax(old_scores / 0.2)
extended_prototypes = np.vstack([prototypes, outside])
new_probability = softmax(outside @ extended_prototypes.T / 0.2)
print("ngoai hai lop, cosine:", old_scores.round(4).tolist())
print("truoc them lop:", old_probability.round(4).tolist(),
      "lop thang:", int(old_probability.argmax(1)[0]))
print("sau them lop:", new_probability.round(4).tolist(),
      "lop thang:", int(new_probability.argmax(1)[0]))
assert old_scores.max() < 0 and old_probability.max() > 0.98
assert new_probability.argmax(1)[0] == 2

temperature: 1.0 softmax: [[0.5498, 0.4502]]
temperature: 0.2 softmax: [[0.7311, 0.2689]]
ngoai hai lop, cosine: [[-0.995, -0.0995]]
truoc them lop: [[0.0112, 0.9888]] lop thang: 1
sau them lop: [[0.0, 0.0041, 0.9959]] lop thang: 2


Cùng cosine 0.8 và 0.6, temperature 1 cho lớp đầu 0.5498, còn temperature 0.2 cho 0.7311. Temperature chỉ làm model tỏ ra chắc chắn hơn hay kém hơn, không làm nó đúng hơn.

Truy vấn thứ hai không giống lớp nào, cosine với cả hai đều âm. Thế mà softmax vẫn cho lớp 1 tới 0.9888. Softmax chỉ chia điểm giữa những ứng viên đang có, nên bao giờ cũng có một lớp “thắng” với xác suất cao. Thêm một prototype cùng hướng với truy vấn, lớp mới thắng ngay. Xác suất 0.99 trong tập đóng không chứng minh mẫu thật sự thuộc một trong các lớp ban đầu.

## Ví dụ 3: Tổng log và trung bình log chọn hai chuỗi khác nhau

Khi chấm một chuỗi token, ta nhân xác suất có điều kiện của từng token, hay cộng log của chúng. Bảng dưới đây tự đặt xác suất cho hai chuỗi. Chuỗi A có hai token, xác suất `(0.6, 0.6)`. Chuỗi B có ba token, xác suất `(0.4, 0.8, 0.8)`. Đây là bảng để tính tay, chưa phải output của model ngôn ngữ nào.

In [ ]:
import numpy as np

paths = {"A": np.array([0.6, 0.6]), "B": np.array([0.4, 0.8, 0.8])}
total, average = {}, {}
for name, probabilities in paths.items():
    total[name] = np.log(probabilities).sum()
    average[name] = total[name] / len(probabilities)
    perplexity = np.exp(-average[name])
    independent = (1 / probabilities.prod())**(1 / len(probabilities))
    assert np.isclose(perplexity, independent)
    print(name, "P(chuoi):", round(float(probabilities.prod()), 3),
          "sum log:", round(float(total[name]), 4),
          "mean log:", round(float(average[name]), 4),
          "PPL:", round(float(perplexity), 4))
print("tong log chon:", max(total, key=total.get))
print("mean log chon:", max(average, key=average.get))

A P(chuoi): 0.36 sum log: -1.0217 mean log: -0.5108 PPL: 1.6667
B P(chuoi): 0.256 sum log: -1.3626 mean log: -0.4542 PPL: 1.5749
tong log chon: A
mean log chon: B


Tổng log chọn A, vì xác suất cả chuỗi của A là 0.36, lớn hơn 0.256 của B. Chuỗi dài hơn phải nhân thêm một xác suất nhỏ hơn 1, nên tổng log luôn thiên về chuỗi ngắn. Trung bình log chia cho số token, và chọn B, vì tính trung bình mỗi token của B được dự đoán tốt hơn.

Perplexity, viết tắt PPL, là nghịch đảo của trung bình hình học các xác suất token. PPL thấp nghĩa là model ít “bất ngờ” với chuỗi ấy, chứ không có nghĩa nội dung chuỗi đó đúng. Chọn cách chuẩn hóa nào là chọn tiêu chí xếp hạng, nên phải làm theo đúng yêu cầu của bài.

## Ví dụ 4: Học một bigram nhỏ, rồi chấm đúng token kế tiếp

Model bigram học bằng cách đếm các cặp token liền nhau trong sáu câu về thời tiết. Nó chỉ nhìn đúng một token ngay trước. Mọi số đếm được cộng thêm 1 để không xác suất nào bằng 0. Đây là một model ngôn ngữ thống kê rất nhỏ, học tại chỗ. Với lời mở đầu `BOS troi`, ta chỉ chấm phần viết tiếp và token kết thúc `EOS`, không cộng điểm của lời mở đầu.

In [ ]:
from collections import Counter
import numpy as np

vocabulary = ["BOS", "troi", "nang", "mua", "am", "lanh", "EOS"]
source = (["BOS", "troi", "nang", "am", "EOS"],) * 3
source += (["BOS", "troi", "mua", "lanh", "EOS"],) * 2
source += (["BOS", "troi", "nang", "lanh", "EOS"],)
pair_counts = Counter()
context_counts = Counter()
for sentence in source:
    for previous, target in zip(sentence[:-1], sentence[1:]):
        pair_counts[previous, target] += 1
        context_counts[previous] += 1

def next_probability(previous, target):
    return (pair_counts[previous, target] + 1) / (context_counts[previous] + len(vocabulary))

prompt = ["BOS", "troi"]
candidates = {"A": ["nang", "am", "EOS"], "B": ["mua", "lanh", "EOS"]}
scores = {}
for name, completion in candidates.items():
    previous_tokens = [prompt[-1]] + completion[:-1]
    pairs = list(zip(previous_tokens, completion))
    probabilities = np.array([next_probability(previous, target) for previous, target in pairs])
    scores[name] = float(np.log(probabilities).sum())
    print(name, "context -> target:", pairs)
    print("P token:", probabilities.round(4).tolist(), "sum log:", round(scores[name], 4))
assert np.isclose(next_probability("troi", "nang"), 5 / 13)
assert np.isclose(sum(next_probability("troi", word) for word in vocabulary), 1)
print("chon:", max(scores, key=scores.get))

A context -> target: [('troi', 'nang'), ('nang', 'am'), ('am', 'EOS')]
P token: [0.3846, 0.3636, 0.4] sum log: -2.8834
B context -> target: [('troi', 'mua'), ('mua', 'lanh'), ('lanh', 'EOS')]
P token: [0.2308, 0.3333, 0.4] sum log: -3.4812
chon: A


Dòng `context -> target` cho thấy cách ghép cặp. Token đầu của phần viết tiếp được dự đoán từ token cuối của lời mở đầu, là `troi`. Mỗi token sau được dự đoán từ token ngay trước nó. Không token nào được dùng để dự đoán chính nó. Lệch một vị trí ở bước ghép cặp là lỗi rất hay gặp khi chấm chuỗi.

Xác suất của `nang` sau `troi` là $5/13$. Trong sáu câu, `troi → nang` xuất hiện bốn lần, cộng 1 thành 5. `troi` xuất hiện sáu lần, cộng thêm 7 là số token trong từ vựng, thành 13. Model chọn chuỗi A, khớp với việc “trời nắng” xuất hiện nhiều hơn trong dữ liệu. Sáu câu thì quá ít để nói model này hiểu tiếng Việt. Ví dụ chỉ nhằm cho thấy cách ghép vị trí và ranh giới giữa lời mở đầu với phần được chấm.